In [20]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
import base64

load_dotenv()

True

In [21]:
with open("blood_test.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

    image_b64[:200]
    print(image_b64)

...


In [22]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_64}"}},
    {"type": "text",       "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
    
])

response = llm.invoke([message])
print(response.content)

**COMPLETE BLOOD COUNT (CBC)**

*   **Hemoglobin:** 15.1 g/dL (Normal: 13.5-17.5) - **Within Normal Range**
*   **Hematocrit:** 44% (Normal: 41-53%) - **Within Normal Range**
*   **WBC:** 6.8 x10^3/μL (Normal: 4.5-11.0) - **Within Normal Range**
*   **Platelets:** 220 x10^3/μL (Normal: 150-400) - **Within Normal Range**

**LIPID PANEL**

*   **Total Cholesterol:** 238 mg/dL (Normal: <200) - **Outside Normal Range (High)**
*   **LDL Cholesterol:** 162 mg/dL (Normal: <100) - **Outside Normal Range (High)**
*   **HDL Cholesterol:** 36 mg/dL (Normal: >40) - **Outside Normal Range (Low)**
*   **Triglycerides:** 188 mg/dL (Normal: <150) - **Outside Normal Range (High)**

**METABOLIC PANEL**

*   **Glucose (Fasting):** 92 mg/dL (Normal: 70-99) - **Within Normal Range**
*   **HbA1c:** 5.3% (Normal: <5.7%) - **Within Normal Range**
*   **Creatinine:** 1.0 mg/dL (Normal: 0.7-1.3) - **Within Normal Range**
*   **eGFR:** 82 mL/min (Normal: >60) - **Within Normal Range**

**LIVER FUNCTION**

*   **

In [29]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
              "eat":        ["fruit", "vegetables", "whole grains", "lean protein"],
             "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed drinks"],
        },
        "high_sugar": {
            "eat":         ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat":  ["white rice", "white sugar", "junk food", "sugar drinks"],
        },
        "normal": {
            "eat":         ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat":  ["excessive sugar", "processed food", "trans fats"],
       },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [32]:
SYSTEM_PROMPT = """
You are a helpful medical and nuutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = (create_agent)(
    llm,
    tools = [get_diet_recommendation],
    system_prompt= SYSTEM_PROMPT,
    #chechpointer= InMemorySaver()
     
)

In [37]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",       "text": "Analyse this blood work report and suggest a diet plan."},
    ])] 
})

print(result["messages"][-1].content)

Based on the blood work report, the patient has high cholesterol. The diet plan for high cholesterol includes eating fruits, vegetables, whole grains, and lean protein, while avoiding red meat, fried food, full-fat dairy, and processed drinks.
